# Lab: Overfitting

**BINF 6210/8210 - Machine Learning for Bioinformatics**
___

This notebook illustrates how logistic regression can **underfit, generalize well, or overfit**, especially when the number of features is large relative to the number of samples.

### Learning objectives
1. Explain why excellent training performance does not guarantee good performance on unseen data.
2. Recognize overfitting from a training–validation performance gap.
3. Explain how logistic-regression regularization affects model complexity.
4. Use cross-validation to evaluate generalization.
5. Examine how irrelevant features can increase overfitting.
6. Use a learning curve to diagnose high variance.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, learning_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, RocCurveDisplay

RANDOM_STATE = 42

## 1. Create a high-dimensional classification problem

High-dimensional biological datasets often contain many measured variables but relatively few samples. Here we simulate **200 samples and 500 features**, but only 10 features are informative. This gives the model many opportunities to fit random patterns.

In [2]:
X, y = make_classification(
    n_samples=200, n_features=500, n_informative=10, n_redundant=10,
    n_classes=2, class_sep=1.0, flip_y=0.05, random_state=RANDOM_STATE
)
print("X shape:", X.shape)
print("Class counts:", np.bincount(y))

X shape: (200, 500)
Class counts: [102  98]


## 2. Create development and test sets

The test set is held aside and is **not used to choose regularization strength**. Within the development set, we use stratified cross-validation.

In [3]:
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)
print("Development samples:", len(y_dev))
print("Test samples:", len(y_test))

Development samples: 150
Test samples: 50


## 3. Logistic regression and regularization

Conceptually, L2-regularized logistic regression minimizes

\[
\text{data-fitting loss}+\lambda\sum_{j=1}^{p}\beta_j^2.
\]

In scikit-learn, `C` is inversely related to regularization strength:

- **small `C`** → strong regularization → simpler model
- **large `C`** → weak regularization → more flexible model

In [4]:
C_values = np.logspace(-5, 5, 15)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = []
for C in C_values:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(C=C, l1_ratio=0.0, solver="lbfgs", max_iter=5000))
    ])
    scores = cross_validate(model, X_dev, y_dev, cv=cv, scoring="roc_auc", return_train_score=True)
    rows.append({
        "C": C,
        "mean_train_auc": scores["train_score"].mean(),
        "mean_validation_auc": scores["test_score"].mean(),
        "std_validation_auc": scores["test_score"].std()
    })
results = pd.DataFrame(rows)
results

,C,mean_train_auc,mean_validation_auc,std_validation_auc
0,0.000010,0.999166,0.709190,0.071548
1,0.000052,0.999166,0.709190,0.071548
2,0.000268,0.999389,0.710071,0.067274
3,0.001389,0.999778,0.702063,0.069492
4,0.007197,1.000000,0.693159,0.068916
5,0.037276,1.000000,0.677127,0.065552
6,0.193070,1.000000,0.676226,0.064287
7,1.000000,1.000000,0.678000,0.066668
8,5.179475,1.000000,0.681560,0.065955
9,26.826958,1.000000,0.682452,0.070353


## 4. Training versus validation performance

For a metric where larger is better, define

\[
\text{generalization gap}
=
\text{training performance}-\text{validation performance}.
\]

A large positive gap is evidence that the model may be overfitting.

In [ ]:
results["generalization_gap"] = results["mean_train_auc"] - results["mean_validation_auc"]

fig, ax = plt.subplots(figsize=(9, 6))
ax.semilogx(results["C"], results["mean_train_auc"], marker="o", label="Training AUROC")
ax.semilogx(results["C"], results["mean_validation_auc"], marker="o", label="Validation AUROC")
ax.set_xlabel("C (larger C = weaker regularization)")
ax.set_ylabel("Mean AUROC")
ax.set_title("Training vs. Validation Performance")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

### Questions for discussion
1. What happens to training AUROC as `C` increases?
2. Does validation AUROC behave the same way?
3. Where is the training–validation gap largest?
4. Why can weaker regularization improve training performance without improving validation performance?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogx(results["C"], results["generalization_gap"], marker="o")
ax.axhline(0, linestyle="--", linewidth=1)
ax.set_xlabel("C (larger C = weaker regularization)")
ax.set_ylabel("Training AUROC − Validation AUROC")
ax.set_title("Generalization Gap")
ax.grid(alpha=0.25)
plt.show()

## 5. Inspect coefficient magnitude

As regularization becomes weaker, logistic regression has more freedom to assign large coefficients to features. With many features and few samples, some apparent associations arise by chance.

In [ ]:
coef_rows = []
for C in C_values:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(C=C, l1_ratio=0.0, solver="lbfgs", max_iter=5000))
    ])
    model.fit(X_dev, y_dev)
    coef = model.named_steps["logreg"].coef_[0]
    coef_rows.append({"C": C, "L2_norm": np.linalg.norm(coef),
                      "max_abs_coefficient": np.abs(coef).max()})

coef_results = pd.DataFrame(coef_rows)

fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogx(coef_results["C"], coef_results["L2_norm"], marker="o")
ax.set_xlabel("C (larger C = weaker regularization)")
ax.set_ylabel("L2 norm of coefficient vector")
ax.set_title("Weaker Regularization Allows Larger Coefficients")
ax.grid(alpha=0.25)
plt.show()

## 6. Select `C` using cross-validation

Choose `C` using the development data only. The untouched test set is not involved in this decision.

In [ ]:
best_idx = results["mean_validation_auc"].idxmax()
best_C = results.loc[best_idx, "C"]
print(f"Best C based on cross-validation: {best_C:.5g}")
print(f"Mean CV AUROC: {results.loc[best_idx, 'mean_validation_auc']:.3f}")

## 7. Compare strong, selected, and very weak regularization

Each model is now fitted to the complete development set and evaluated on the untouched test set.

In [ ]:
C_compare = {
    "Strong regularization": C_values[0],
    "CV-selected": best_C,
    "Very weak regularization": C_values[-1]
}

comparison, fitted_models = [], {}
for name, C in C_compare.items():
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(C=C, l1_ratio=0.0, solver="lbfgs", max_iter=5000))
    ])
    model.fit(X_dev, y_dev)
    fitted_models[name] = model
    train_prob = model.predict_proba(X_dev)[:, 1]
    test_prob = model.predict_proba(X_test)[:, 1]
    comparison.append({
        "Model": name, "C": C,
        "Training AUROC": roc_auc_score(y_dev, train_prob),
        "Test AUROC": roc_auc_score(y_test, test_prob),
        "Training accuracy": accuracy_score(y_dev, model.predict(X_dev)),
        "Test accuracy": accuracy_score(y_test, model.predict(X_test))
    })

comparison_df = pd.DataFrame(comparison)
comparison_df

### Interpretation

Do not identify overfitting simply because training performance is high. The concern is **failure to generalize**:

\[
\boxed{\text{Overfitting is a failure to generalize}}
\]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for name, model in fitted_models.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, name=name, ax=ax)
ax.plot([0, 1], [0, 1], linestyle="--")
ax.set_title("ROC Curves on the Untouched Test Set")
plt.show()

## 8. What happens when we add irrelevant features?

This experiment varies dimensionality while keeping only 10 informative and 10 redundant features. It illustrates the challenge of \(p \gg n\) common in omics studies.

In [ ]:
feature_counts = [20, 50, 100, 250, 500, 1000]
noise_rows = []

for n_features in feature_counts:
    X_tmp, y_tmp = make_classification(
        n_samples=200, n_features=n_features, n_informative=10, n_redundant=10,
        n_classes=2, class_sep=1.0, flip_y=0.05, random_state=RANDOM_STATE
    )
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(C=1000, l1_ratio=0.0, solver="lbfgs", max_iter=5000))
    ])
    scores = cross_validate(model, X_tmp, y_tmp, cv=cv, scoring="roc_auc", return_train_score=True)
    noise_rows.append({
        "Number of features": n_features,
        "Training AUROC": scores["train_score"].mean(),
        "Validation AUROC": scores["test_score"].mean()
    })

noise_results = pd.DataFrame(noise_rows)
noise_results

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(noise_results["Number of features"], noise_results["Training AUROC"],
        marker="o", label="Training AUROC")
ax.plot(noise_results["Number of features"], noise_results["Validation AUROC"],
        marker="o", label="Validation AUROC")
ax.set_xscale("log")
ax.set_xlabel("Number of features")
ax.set_ylabel("Mean AUROC")
ax.set_title("High Dimensionality and Overfitting")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## 9. Learning curve

A learning curve shows how training and validation performance change as the training sample size increases. A persistent training–validation gap is characteristic of high variance.

In [ ]:
overfit_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(C=1000, l1_ratio=0.0, solver="lbfgs", max_iter=5000))
])

train_sizes, train_scores, val_scores = learning_curve(
    overfit_model, X, y, cv=cv, scoring="roc_auc",
    train_sizes=np.linspace(0.2, 1.0, 6),
    shuffle=True, random_state=RANDOM_STATE
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sizes, train_scores.mean(axis=1), marker="o", label="Training AUROC")
ax.plot(train_sizes, val_scores.mean(axis=1), marker="o", label="Validation AUROC")
ax.set_xlabel("Number of training samples")
ax.set_ylabel("Mean AUROC")
ax.set_title("Learning Curve for Weakly Regularized Logistic Regression")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## 10. Summary

Logistic regression can overfit when the number of predictors is large relative to the number of samples, predictors contain substantial noise, or regularization is too weak.

Ways to reduce or detect overfitting include appropriate regularization, cross-validation, scientifically justified feature reduction, larger training sets, and keeping the final test set untouched during model development.

\[
\boxed{\text{Training performance} \neq \text{generalization performance}}
\]

> **The goal is not to fit the training data as closely as possible; it is to learn patterns that generalize to unseen data.**

## 11. Exercises

1. Change the number of samples from 200 to 1,000. How does the training–validation gap change?
2. Change the number of features from 500 to 50. What happens?
3. Increase `flip_y` from 0.05 to 0.20. How does label noise affect performance?
4. Compare `C=0.001`, `C=1`, and `C=1000` in terms of regularization and generalization.
5. Why would selecting `C` using the final test-set AUROC compromise the role of the test set?